# DNAscape data management

A compact workflow for getting data into and out of DNAscape.

The notebook moves in the order users usually need: load data, manipulate a DNAscape-ready profile, save it, then convert common sequencing tracks into the same ready format.

## Sections

1. Setup and paths
2. Load DNAscape-ready data
3. Manipulate a profile
4. Save prepared data
5. Convert sequencing tracks
6. Ready-to-use checklist

## 1. Setup and paths

Paths can be absolute, relative to the notebook, relative to the repository root, or use `~`. The bundled H1 replication-timing table is only a default example.

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dnascape import *


def resolve_path(path=None, default=None, must_exist=False):
    """Resolve absolute, home-relative, notebook-relative, or repo-relative paths."""
    if path in (None, ""):
        path = default
    if path in (None, ""):
        raise ValueError("Provide a path or a default path.")

    raw = Path(path).expanduser()
    candidates = [raw] if raw.is_absolute() else [Path.cwd() / raw, PROJECT_ROOT / raw]
    candidates = [candidate.resolve() for candidate in candidates]

    for candidate in candidates:
        if candidate.exists():
            return candidate
    if must_exist:
        raise FileNotFoundError(f"Could not find {path!r} from notebook or repo root.")
    return candidates[0]


DEFAULT_TIMING_PATH = PROJECT_ROOT / "data" / "replication_timing" / "replication_timing_H1.csv"

## 2. Load DNAscape-ready data

### Tables: CSV, TSV, TXT, or GZ

Use a table when a file contains chromosomes, positions, samples, or multiple signals. `sep=None` lets pandas infer commas versus tabs, which is useful for exported genomic tracks.

In [ ]:
USER_INPUT_PATH = None  # e.g. "~/tracks/repliseq_chr1.tsv" or r"D:/data/track.csv"
input_path = resolve_path(USER_INPUT_PATH, default=DEFAULT_TIMING_PATH, must_exist=True)

table = pd.read_csv(input_path, sep=None, engine="python")
chromosome = "chr1"
profile = table[chromosome].dropna().to_numpy(dtype=float)

table.head(), profile.shape

### Vectors: one value per bin

Use a vector when the file already has exactly one numeric value per genomic bin.

In [ ]:
# VECTOR_INPUT_PATH = "path/to/profile.txt"
# vector_path = resolve_path(VECTOR_INPUT_PATH, must_exist=True)
# profile = np.loadtxt(vector_path, dtype=float)

## 3. Manipulate a profile

### Select and resample

In [ ]:
raw_resolution_kb = 1
interval_kb = (0, 60_000)
output_resolution_kb = 5

profile_region, positions_kb = select_profile_region(
    profile,
    raw_resolution_kb=raw_resolution_kb,
    interval_kb=interval_kb,
    output_resolution_kb=output_resolution_kb,
    resample_method="mean",
)

ready_profile = pd.DataFrame(
    {
        "position_kb": positions_kb,
        "replication_timing": profile_region,
    }
)

ready_profile.head()

### Clip, rescale, and smooth

In [ ]:
def minmax(values, out_range=(0.0, 1.0)):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.full_like(arr, np.nan)
    lo, hi = np.nanmin(arr[finite]), np.nanmax(arr[finite])
    if np.isclose(lo, hi):
        return np.full_like(arr, out_range[0], dtype=float)
    scaled = (arr - lo) / (hi - lo)
    return scaled * (out_range[1] - out_range[0]) + out_range[0]


def rolling_mean(values, window_bins=11):
    return (
        pd.Series(values, dtype="float64")
        .rolling(window=int(window_bins), center=True, min_periods=1)
        .mean()
        .to_numpy()
    )

In [ ]:
timing = ready_profile["replication_timing"].to_numpy()

ready_profile["timing_clipped"] = np.clip(timing, 0, 630)
ready_profile["timing_0_1"] = minmax(ready_profile["timing_clipped"])
ready_profile["timing_smoothed"] = rolling_mean(ready_profile["timing_clipped"], window_bins=21)

ready_profile.head()

## 4. Save prepared data

### Save as a table

In [ ]:
USER_OUTPUT_TABLE = PROJECT_ROOT / "data" / "output" / "timing_prepared.tsv"
output_table = resolve_path(USER_OUTPUT_TABLE)
output_table.parent.mkdir(parents=True, exist_ok=True)

ready_profile.to_csv(output_table, sep="\t", index=False)
output_table

### Save as a vector

In [ ]:
USER_OUTPUT_VECTOR = PROJECT_ROOT / "data" / "output" / "timing_smoothed.txt"
output_vector = resolve_path(USER_OUTPUT_VECTOR)
output_vector.parent.mkdir(parents=True, exist_ok=True)

np.savetxt(output_vector, ready_profile["timing_smoothed"].to_numpy())
output_vector

## 5. Convert sequencing tracks

Many sequencing formats are interval tracks: `chrom`, `start`, `end`, `signal`. The helper below averages interval signal into fixed genomic bins and returns a DNAscape-ready table.

In [ ]:
def interval_track_to_ready_table(
    track,
    chrom,
    start_bp,
    end_bp,
    bin_size_bp,
    value_col="signal",
    output_col="signal",
):
    starts = np.arange(int(start_bp), int(end_bp), int(bin_size_bp), dtype=int)
    ends = np.minimum(starts + int(bin_size_bp), int(end_bp))
    weighted = np.zeros(starts.size, dtype=float)
    covered = np.zeros(starts.size, dtype=float)

    selected = track.loc[
        (track["chrom"] == chrom)
        & (track["end"] > start_bp)
        & (track["start"] < end_bp),
        ["start", "end", value_col],
    ]

    for interval_start, interval_end, value in selected.itertuples(index=False, name=None):
        overlap = np.clip(
            np.minimum(ends, int(interval_end)) - np.maximum(starts, int(interval_start)),
            0,
            None,
        )
        weighted += overlap * float(value)
        covered += overlap

    signal = np.divide(
        weighted,
        covered,
        out=np.full(starts.size, np.nan, dtype=float),
        where=covered > 0,
    )

    return pd.DataFrame(
        {
            "chrom": chrom,
            "start_bp": starts,
            "end_bp": ends,
            "position_kb": starts / 1_000,
            output_col: signal,
        }
    )


def load_interval_track(path=None, fallback=None):
    """Load a BED/bedGraph-like table, or return a small fallback example."""
    if path in (None, ""):
        if fallback is None:
            raise ValueError("Provide an interval-track path or fallback data.")
        return fallback.copy()

    track = pd.read_csv(resolve_path(path, must_exist=True), sep=None, engine="python", comment="#")
    required = {"chrom", "start", "end", "signal"}
    if required.issubset(track.columns):
        return track

    track = track.iloc[:, :4].copy()
    track.columns = ["chrom", "start", "end", "signal"]
    return track

### Repli-seq, BrdU-seq, EdU-seq

Use these tracks as replicated fraction, replication timing, or an enrichment profile after choosing the correct biological interpretation for the experiment.

In [ ]:
USER_REPLISEQ_PATH = None  # e.g. "~/tracks/repliseq_chr1.bedgraph"

repliseq_example = pd.DataFrame(
    {
        "chrom": ["chr1", "chr1", "chr1", "chr1"],
        "start": [0, 2_500, 5_000, 9_000],
        "end": [2_500, 5_000, 9_000, 12_000],
        "signal": [0.20, 0.70, 0.40, 1.00],
    }
)
repliseq_track = load_interval_track(USER_REPLISEQ_PATH, fallback=repliseq_example)

repliseq_ready = interval_track_to_ready_table(
    repliseq_track,
    chrom="chr1",
    start_bp=0,
    end_bp=12_000,
    bin_size_bp=1_000,
    output_col="replicated_fraction",
)

repliseq_ready.head()

### SNS-seq, ini-seq, bubble-seq

Use these tracks as origin or initiation signal. A simple first pass is to bin the signal and rescale it to 0 to 1.

In [ ]:
USER_SNS_PATH = None  # e.g. "~/tracks/sns_seq_peaks.bedgraph"

sns_example = pd.DataFrame(
    {
        "chrom": ["chr1", "chr1", "chr1"],
        "start": [0, 4_000, 8_000],
        "end": [4_000, 8_000, 12_000],
        "signal": [12.0, 3.0, 8.0],
    }
)
sns_track = load_interval_track(USER_SNS_PATH, fallback=sns_example)

origin_ready = interval_track_to_ready_table(
    sns_track,
    chrom="chr1",
    start_bp=0,
    end_bp=12_000,
    bin_size_bp=1_000,
    output_col="origin_signal",
)
origin_ready["origin_signal"] = minmax(origin_ready["origin_signal"])

origin_ready.head()

### OK-seq, FORK-seq

Use these tracks as fork directionality or strand-bias signal. Keep the sign and clip to the expected range when appropriate.

In [ ]:
USER_OKSEQ_PATH = None  # e.g. "~/tracks/okseq_rfd.bedgraph"

okseq_example = pd.DataFrame(
    {
        "chrom": ["chr1", "chr1", "chr1"],
        "start": [0, 4_000, 8_000],
        "end": [4_000, 8_000, 12_000],
        "signal": [-0.8, 0.1, 0.9],
    }
)
okseq_track = load_interval_track(USER_OKSEQ_PATH, fallback=okseq_example)

fork_directionality_ready = interval_track_to_ready_table(
    okseq_track,
    chrom="chr1",
    start_bp=0,
    end_bp=12_000,
    bin_size_bp=1_000,
    output_col="fork_directionality",
)
fork_directionality_ready["fork_directionality"] = np.clip(
    fork_directionality_ready["fork_directionality"],
    -1,
    1,
)

fork_directionality_ready.head()

### BigWig

BigWig follows the same pattern: query mean signal over fixed bins, then save the resulting table. Use `pyBigWig` or UCSC tools upstream, then load the binned table here.

## 6. Ready-to-use checklist

Before using a profile in `rmap` or `rsim`, confirm:

- one value corresponds to one genomic bin;
- `resolution_x` is known in kb;
- chromosome and interval are recorded;
- missing values and smoothing choices are intentional;
- the saved file reloads to the same values.